In [16]:

import numpy as np
from typing import Tuple

def load_data(filepath: str) -> Tuple[np.ndarray, np.ndarray]:
    data = np.loadtxt(filepath, delimiter=",", skiprows=1)
    X = data[:, :-1]
    y = data[:, -1]
    return X, y

def get_true_positives(y_true: np.ndarray, y_pred: np.ndarray) -> int:
    return int(np.sum((y_true == 1) & (y_pred == 1)))

def get_false_positives(y_true: np.ndarray, y_pred: np.ndarray) -> int:
    return int(np.sum((y_true == 0) & (y_pred == 1)))

def get_false_negatives(y_true: np.ndarray, y_pred: np.ndarray) -> int:
    return int(np.sum((y_true == 1) & (y_pred == 0)))

def get_true_negatives(y_true: np.ndarray, y_pred: np.ndarray) -> int:
    return int(np.sum((y_true == 0) & (y_pred == 0)))

def get_precision(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    tp = get_true_positives(y_true, y_pred)
    fp = get_false_positives(y_true, y_pred)
    denom = tp + fp
    if denom == 0:
        return 0.0
    return float(tp / denom)

def get_recall(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    tp = get_true_positives(y_true, y_pred)
    fn = get_false_negatives(y_true, y_pred)
    denom = tp + fn
    if denom == 0:
        return 0.0
    return float(tp / denom)

def get_f1_score(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    prec = get_precision(y_true, y_pred)
    rec = get_recall(y_true, y_pred)
    denom = prec + rec
    if denom == 0:
        return 0.0
    return float(2.0 * (prec * rec) / denom)

def get_accuracy(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean(y_true == y_pred))


Overwriting utils.py


In [18]:

import numpy as np
from typing import Optional, Dict, Union

class LeafNode:
    def __init__(self, predicted_class: int, n_samples: int) -> None:
        self.predicted_class = predicted_class
        self.n_samples = n_samples

class DecisionNode:
    def __init__(
        self,
        feature_index: int,
        threshold: float,
        left: Union["DecisionNode", LeafNode] = None,
        right: Union["DecisionNode", LeafNode] = None,
        n_samples: int = 0,
    ) -> None:
        self.feature_index = feature_index
        self.threshold = threshold
        self.left = left
        self.right = right
        self.n_samples = n_samples

def gini_impurity(y: np.ndarray) -> float:
    if len(y) == 0:
        return 0.0
    _, counts = np.unique(y, return_counts=True)
    p = counts / counts.sum()
    return float(1.0 - np.sum(p ** 2))

def find_best_split(X: np.ndarray, y: np.ndarray) -> Optional[Dict]:
    N, D = X.shape
    best_cost = np.inf
    best_split = None

    for j in range(D):
        thresholds = np.unique(X[:, j])
        for t in thresholds:
            mask = X[:, j] <= t
            y_left = y[mask]
            y_right = y[~mask]

            if len(y_left) == 0 or len(y_right) == 0:
                continue

            cost = (len(y_left) / N) * gini_impurity(y_left) + (len(y_right) / N) * gini_impurity(y_right)

            if cost < best_cost:
                best_cost = cost
                best_split = {
                    "feature_index": int(j),
                    "threshold": float(t),
                    "cost": float(cost)
                }

    return best_split

class DecisionTree:
    def __init__(
        self,
        max_depth: Optional[int] = None,
        min_samples_split: int = 2,
    ) -> None:
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root: Optional[Union[DecisionNode, LeafNode]] = None

    @staticmethod
    def _majority_class(y: np.ndarray) -> int:
        y_int = y.astype(int)
        counts = np.bincount(y_int)
        return int(np.argmax(counts))

    def _grow_tree(
        self,
        X: np.ndarray,
        y: np.ndarray,
        depth: int = 0,
    ) -> Union[DecisionNode, LeafNode]:
        N = len(y)

        # 1. Max depth reached
        if self.max_depth is not None and depth >= self.max_depth:
            return LeafNode(self._majority_class(y), N)

        # 2. Pure node (all labels identical)
        if len(np.unique(y)) == 1:
            return LeafNode(self._majority_class(y), N)

        # 3. Too few samples to split
        if N < self.min_samples_split:
            return LeafNode(self._majority_class(y), N)

        best = find_best_split(X, y)

        # 4. No valid split found
        if best is None:
            return LeafNode(self._majority_class(y), N)

        j = best["feature_index"]
        t = best["threshold"]

        mask = X[:, j] <= t
        left_child = self._grow_tree(X[mask], y[mask], depth + 1)
        right_child = self._grow_tree(X[~mask], y[~mask], depth + 1)

        return DecisionNode(feature_index=j, threshold=t, left=left_child, right=right_child, n_samples=N)

    def fit(self, X: np.ndarray, y: np.ndarray) -> None:
        y_int = y.astype(int)
        self.root = self._grow_tree(X, y_int, depth=0)

    def _predict_single(
        self,
        x: np.ndarray,
        node: Union[DecisionNode, LeafNode],
    ) -> int:
        if isinstance(node, LeafNode):
            return node.predicted_class

        if x[node.feature_index] <= node.threshold:
            return self._predict_single(x, node.left)
        else:
            return self._predict_single(x, node.right)

    def predict(self, X: np.ndarray) -> np.ndarray:
        preds = [self._predict_single(x, self.root) for x in X]
        return np.array(preds)

    def get_depth(self) -> int:
        if self.root is None:
            return -1

        def _depth(node: Union[DecisionNode, LeafNode]) -> int:
            if isinstance(node, LeafNode):
                return 0
            return 1 + max(_depth(node.left), _depth(node.right))

        return _depth(self.root)

    def get_n_leaves(self) -> int:
        if self.root is None:
            return 0

        def _count_leaves(node: Union[DecisionNode, LeafNode]) -> int:
            if isinstance(node, LeafNode):
                return 1
            return _count_leaves(node.left) + _count_leaves(node.right)

        return _count_leaves(self.root)


Overwriting cart.py
